# 03 — Spark SQL: transacciones bancarias (`bank_transactions.csv`)

**Objetivo:** mostrar que Spark SQL y la DataFrame API son la misma cosa por debajo
(comparten el optimizador Catalyst), usando window functions — el puente directo con
SQL distribuido (BigQuery/Hive) de la Sesión 2.

**Dataset:** `bank_transactions.csv` (~7.5 GB) — el dataset transversal del curso (ver
`recursos/datasets/README.md`, Sección 3). Columnas: `transaction_id`, `timestamp`,
`from_*`, `to_*`, `amount`, `currency`, `is_suspicious`, `suspicious_pattern`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("03_spark_sql").getOrCreate()

RUTA = "gs://<TU-BUCKET>/raw/bank_transactions/bank_transactions.csv"

In [ ]:
df = spark.read.csv(RUTA, header=True, inferSchema=True)
df.createOrReplaceTempView("transacciones")
df.printSchema()

## Consulta base: transacciones sospechosas por patrón y moneda

In [ ]:
resumen_sql = spark.sql("""
    SELECT suspicious_pattern, currency,
           COUNT(*) AS num_transacciones,
           SUM(amount) AS monto_total,
           AVG(amount) AS monto_promedio
    FROM transacciones
    WHERE is_suspicious = true
    GROUP BY suspicious_pattern, currency
    ORDER BY monto_total DESC
""")

print("=== Comparar este plan con el de 02_dataframes.ipynb: mismo tipo de optimización ===")
resumen_sql.explain(mode="formatted")
resumen_sql.show(20, truncate=False)

## Window function: ranking de montos sospechosos por cuenta origen (`from_account`)

Mismo patrón analítico que se usa en `recursos/hive/hive-queries.sql` (Sección 2.4) —
a propósito, para que quede claro que el SQL analítico es prácticamente idéntico entre
Hive y Spark SQL.

In [ ]:
ranking_sql = spark.sql("""
    SELECT transaction_id, from_account, amount, suspicious_pattern,
           RANK() OVER (PARTITION BY from_account ORDER BY amount DESC) AS ranking
    FROM transacciones
    WHERE is_suspicious = true
""")

top_por_cuenta = ranking_sql.filter("ranking <= 3")
top_por_cuenta.show(30, truncate=False)

In [ ]:
spark.stop()